In [0]:
%pip install boto3
dbutils.library.restartPython()

In [0]:
# ===================================================================
# Export Fantasy News to Cloudflare R2
# ===================================================================
# This notebook exports enriched fantasy news data from Unity Catalog
# to Cloudflare R2 for frontend consumption via S3-compatible API

import json
from datetime import datetime
import boto3

print("🚀 FantasAI News Export to R2")
print("=" * 70)

# === Configuration ===
CATALOG = "main"
SCHEMA = "fantasai"

# Retrieve R2 credentials from Databricks secrets
R2_ACCESS_KEY_ID = dbutils.secrets.get(scope="r2_credentials", key="r2_access_key_id")
R2_SECRET_ACCESS_KEY = dbutils.secrets.get(scope="r2_credentials", key="r2_secret_access_key")
R2_ENDPOINT_URL = dbutils.secrets.get(scope="r2_credentials", key="r2_endpoint_url")
R2_BUCKET_NAME = dbutils.secrets.get(scope="r2_credentials", key="r2_bucket_name")

print(f"✓ Catalog: {CATALOG}")
print(f"✓ Schema: {SCHEMA}")
print(f"✓ R2 Bucket: {R2_BUCKET_NAME}")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

In [0]:
# Initialize R2 S3 client
s3_client = boto3.client(
    's3',
    endpoint_url=R2_ENDPOINT_URL,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name='auto'
)

print("✓ R2 client initialized")

def export_to_r2(r2_key, data_dict, description):
    """
    Export data to R2 via S3-compatible API.
    
    Args:
        r2_key: Full R2 object key (e.g., "fantasai/news/player_notes.json")
        data_dict: Dictionary to serialize as JSON
        description: Human-readable description for logging
    
    Returns:
        Tuple of (success: bool, message: str)
    """
    try:
        # Serialize to JSON
        json_payload = json.dumps(data_dict, default=str, indent=2)
        payload_size_kb = len(json_payload.encode('utf-8')) / 1024
        
        print(f"\n📤 Uploading {description}...")
        print(f"   R2 Key: {r2_key}")
        print(f"   Size: {payload_size_kb:.1f} KB")
        print(f"   Records: {len(data_dict.get('data', []))}")
        
        # Upload to R2 via S3 API
        s3_client.put_object(
            Bucket=R2_BUCKET_NAME,
            Key=r2_key,
            Body=json_payload.encode('utf-8'),
            ContentType='application/json'
        )
        
        print(f"   ✅ Success")
        return True, "Uploaded successfully"
            
    except Exception as e:
        error_msg = f"Upload failed: {str(e)[:200]}"
        print(f"   ❌ {error_msg}")
        return False, error_msg

print("✅ Helper functions loaded")

In [0]:
print("\n" + "=" * 70)
print("📋 EXPORTING PLAYER NOTES")
print("=" * 70)

# Read from Delta table
player_notes_df = spark.sql("""
    SELECT 
        player_id,
        player_name,
        position,
        team,
        notes,
        overall_sentiment,
        overall_impact_score,
        has_critical_news,
        has_injury_concern,
        has_opportunity_change,
        note_count,
        last_updated,
        updated_at
    FROM main.fantasai.gold_player_notes
    ORDER BY overall_impact_score DESC, last_updated DESC
""")

print(f"✅ Read {player_notes_df.count()} player notes from Unity Catalog")

# Convert to pandas for JSON serialization
player_notes_pd = player_notes_df.toPandas()

# Transform to frontend schema
player_notes_data = []
for _, row in player_notes_pd.iterrows():
    player_notes_data.append({
        "player_id": row['player_id'],
        "player_name": row['player_name'],
        "position": row['position'],
        "team": row['team'],
        "notes": row['notes'],
        "overall_sentiment": row['overall_sentiment'],
        "overall_impact_score": float(row['overall_impact_score']) if row['overall_impact_score'] is not None else 0.0,
        "has_critical_news": bool(row['has_critical_news']),
        "has_injury_concern": bool(row['has_injury_concern']),
        "has_opportunity_change": bool(row['has_opportunity_change']),
        "note_count": int(row['note_count']),
        "last_updated": row['last_updated'].isoformat() if row['last_updated'] is not None else None,
        "updated_at": row['updated_at'].isoformat() if row['updated_at'] is not None else None
    })

# Prepare payload
payload = {
    "data": player_notes_data,
    "metadata": {
        "total_players": len(player_notes_data),
        "exported_at": datetime.utcnow().isoformat() + "Z",
        "source": "databricks"
    }
}

# Export to R2
success, message = export_to_r2(
    "fantasai/news/player_notes.json",
    payload,
    "Player Notes"
)

if not success:
    raise Exception(f"Failed to export player notes: {message}")

In [0]:
print("\n" + "=" * 70)
print("🤖 EXPORTING AI SUMMARIES")
print("=" * 70)

# Read from Delta table
ai_summaries_df = spark.sql("""
    SELECT 
        ai.summary_id,
        ai.news_id,
        ai.summary_text,
        ai.fantasy_insight,
        ai.fantasy_relevance_score,
        ai.impact_category,
        ai.priority_level,
        ai.impacted_players,
        ai.is_time_sensitive,
        ai.llm_model,
        ai.llm_tokens_used,
        ai.generated_at,
        en.headline,
        en.source_url,
        en.published_at,
        en.mentioned_teams
    FROM main.fantasai.gold_news_ai_summaries ai
    INNER JOIN main.fantasai.gold_enriched_news en
        ON ai.news_id = en.news_id
    ORDER BY ai.generated_at DESC
    LIMIT 100
""")

print(f"✅ Read {ai_summaries_df.count()} AI summaries from Unity Catalog")

# Convert to pandas
ai_summaries_pd = ai_summaries_df.toPandas()

# Transform to frontend schema
ai_summaries_data = []
for _, row in ai_summaries_pd.iterrows():
    ai_summaries_data.append({
        "summary_id": row['summary_id'],
        "news_id": row['news_id'],
        "headline": row['headline'],
        "source_url": row['source_url'],
        "summary_text": row['summary_text'],
        "fantasy_insight": row['fantasy_insight'],
        "fantasy_relevance_score": float(row['fantasy_relevance_score']) if row['fantasy_relevance_score'] is not None else 0.0,
        "impact_category": row['impact_category'],
        "priority_level": row['priority_level'],
        "impacted_players": row['impacted_players'],
        "mentioned_teams": row['mentioned_teams'],
        "is_time_sensitive": bool(row['is_time_sensitive']),
        "llm_model": row['llm_model'],
        "llm_tokens_used": int(row['llm_tokens_used']) if row['llm_tokens_used'] is not None else 0,
        "published_at": row['published_at'].isoformat() if row['published_at'] is not None else None,
        "generated_at": row['generated_at'].isoformat() if row['generated_at'] is not None else None
    })

# Prepare payload
payload = {
    "data": ai_summaries_data,
    "metadata": {
        "total_summaries": len(ai_summaries_data),
        "exported_at": datetime.utcnow().isoformat() + "Z",
        "source": "databricks"
    }
}

# Export to R2
success, message = export_to_r2(
    "fantasai/news/ai_summaries.json",
    payload,
    "AI Summaries"
)

if not success:
    raise Exception(f"Failed to export AI summaries: {message}")

In [0]:
print("\n" + "=" * 70)
print("📰 EXPORTING ENRICHED NEWS")
print("=" * 70)

# Read from Delta table
enriched_news_df = spark.sql("""
    SELECT 
        news_id,
        headline,
        source_table as source_name,
        source_url,
        full_text,
        primary_player_id,
        mentioned_players,
        mentioned_teams,
        extraction_confidence,
        published_at,
        enriched_at
    FROM main.fantasai.gold_enriched_news
    ORDER BY enriched_at DESC
    LIMIT 200
""")

print(f"✅ Read {enriched_news_df.count()} enriched articles from Unity Catalog")

# Convert to pandas
enriched_news_pd = enriched_news_df.toPandas()

# Transform to frontend schema
enriched_news_data = []
for _, row in enriched_news_pd.iterrows():
    enriched_news_data.append({
        "news_id": row['news_id'],
        "headline": row['headline'],
        "source_name": row['source_name'],
        "source_url": row['source_url'],
        "full_text": row['full_text'],
        "primary_player_id": row['primary_player_id'],
        "mentioned_players": row['mentioned_players'],
        "mentioned_teams": row['mentioned_teams'],
        "extraction_confidence": float(row['extraction_confidence']) if row['extraction_confidence'] is not None else 0.0,
        "published_at": row['published_at'].isoformat() if row['published_at'] is not None else None,
        "enriched_at": row['enriched_at'].isoformat() if row['enriched_at'] is not None else None
    })

# Prepare payload
payload = {
    "data": enriched_news_data,
    "metadata": {
        "total_articles": len(enriched_news_data),
        "exported_at": datetime.utcnow().isoformat() + "Z",
        "source": "databricks"
    }
}

# Export to R2
success, message = export_to_r2(
    "fantasai/news/enriched_news.json",
    payload,
    "Enriched News"
)

if not success:
    raise Exception(f"Failed to export enriched news: {message}")

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime

print("\n" + "=" * 70)
print("🏥 EXPORTING CURRENT PLAYER INJURY STATUS")
print("=" * 70)

# Read from Delta table
silver_player_news_df = spark.sql("""
    SELECT 
        player_id,
        player_name,
        position,
        team,
        news_updated,
        injury_status,
        injury_notes,
        status,
        depth_chart_order,
        depth_chart_position,
        fetched_at
    FROM main.fantasai.silver_player_news
    ORDER BY 
        CASE 
            WHEN injury_status IS NOT NULL THEN 0
            ELSE 1
        END,
        team,
        depth_chart_order
""")

print(f"✅ Read {silver_player_news_df.count()} player records from Unity Catalog")

# Convert to pandas
silver_player_news_pd = silver_player_news_df.toPandas()

# Transform to frontend schema
silver_player_news_data = []
for _, row in silver_player_news_pd.iterrows():
    # Handle depth_chart_order with NaN check
    depth_order = row['depth_chart_order']
    if pd.notna(depth_order):
        depth_order = int(depth_order)
    else:
        depth_order = None
    
    silver_player_news_data.append({
        "player_id": row['player_id'],
        "player_name": row['player_name'],
        "position": row['position'],
        "team": row['team'],
        "news_updated": row['news_updated'].isoformat() if pd.notna(row['news_updated']) else None,
        "injury_status": row['injury_status'] if pd.notna(row['injury_status']) else None,
        "injury_notes": row['injury_notes'] if pd.notna(row['injury_notes']) else None,
        "status": row['status'] if pd.notna(row['status']) else None,
        "depth_chart_order": depth_order,
        "depth_chart_position": row['depth_chart_position'] if pd.notna(row['depth_chart_position']) else None,
        "fetched_at": row['fetched_at'].isoformat() if pd.notna(row['fetched_at']) else None
    })

# Prepare payload
payload = {
    "data": silver_player_news_data,
    "metadata": {
        "total_players": len(silver_player_news_data),
        "injured_players": len([p for p in silver_player_news_data if p['injury_status'] is not None]),
        "exported_at": datetime.utcnow().isoformat() + "Z",
        "source": "databricks",
        "table": "main.fantasai.silver_player_news",
        "note": "Using silver_player_news is appropriate for real-time injury/depth chart overlays"
    }
}

# Export to R2
success, message = export_to_r2(
    "fantasai/players/injury_overlay.json",
    payload,
    "Current Player Injury Status"
)

if not success:
    raise Exception(f"Failed to export player injury status: {message}")

In [0]:
print("\n" + "=" * 70)
print("📰 EXPORTING COMBINED PLAYER NEWS (ALL TIMESTAMPS)")
print("=" * 70)
print("This endpoint combines enriched news + AI summaries + player context")
print("with all three timestamps: published_at, enriched_at, ai_generated_at")
print()

# Read from the combined export table
combined_news_df = spark.sql("""
    SELECT 
        news_id,
        headline,
        source_url,
        full_text,
        player_id,
        player_name,
        position,
        team,
        summary_text,
        fantasy_insight,
        impact_score,
        impact_category,
        published_at,
        enriched_at,
        ai_generated_at
    FROM main.fantasai.export_player_news
    ORDER BY published_at DESC
""")

print(f"✅ Read {combined_news_df.count()} combined news records from Unity Catalog")

# Convert to pandas
combined_news_pd = combined_news_df.toPandas()

# Transform to frontend schema
combined_news_data = []
for _, row in combined_news_pd.iterrows():
    combined_news_data.append({
        "news_id": row['news_id'],
        "headline": row['headline'],
        "source_url": row['source_url'],
        "full_text": row['full_text'],
        "player_id": row['player_id'],
        "player_name": row['player_name'],
        "position": row['position'],
        "team": row['team'],
        "summary_text": row['summary_text'],
        "fantasy_insight": row['fantasy_insight'],
        "impact_score": float(row['impact_score']) if row['impact_score'] is not None else 0.0,
        "impact_category": row['impact_category'],
        # THREE TIMESTAMPS - all included!
        "published_at": row['published_at'].isoformat() if row['published_at'] is not None else None,
        "enriched_at": row['enriched_at'].isoformat() if row['enriched_at'] is not None else None,
        "ai_generated_at": row['ai_generated_at'].isoformat() if row['ai_generated_at'] is not None else None
    })

# Prepare payload with metadata
payload = {
    "data": combined_news_data,
    "metadata": {
        "total_articles": len(combined_news_data),
        "exported_at": datetime.utcnow().isoformat() + "Z",
        "source": "databricks",
        "description": "Combined player news with enriched content, AI summaries, and all timestamps"
    }
}

# Export to R2
success, message = export_to_r2(
    "fantasai/analysis/player_news.json",
    payload,
    "Combined Player News"
)

if not success:
    raise Exception(f"Failed to export combined player news: {message}")

print(f"\n✅ Combined player news exported successfully!")
print(f"   Records: {len(combined_news_data)}")
print(f"   Endpoint: fantasai/analysis/player_news.json")

In [0]:
print("\n" + "=" * 70)
print("✅ EXPORT COMPLETE")
print("=" * 70)

print("\n📊 Export Summary:")
print(f"  • Combined Player News: {len(combined_news_data)} records ⭐ RECOMMENDED")
print(f"  • Player Notes: {len(player_notes_data)} records")
print(f"  • AI Summaries: {len(ai_summaries_data)} records")
print(f"  • Enriched News: {len(enriched_news_data)} records")
print(f"  • Current Injury Status: {len(silver_player_news_data)} records")

print("\n🌐 R2 Objects Created:")
print(f"  • {R2_BUCKET_NAME}/fantasai/analysis/player_news.json ⭐ RECOMMENDED")
print(f"  • {R2_BUCKET_NAME}/fantasai/news/player_notes.json")
print(f"  • {R2_BUCKET_NAME}/fantasai/news/ai_summaries.json")
print(f"  • {R2_BUCKET_NAME}/fantasai/news/enriched_news.json")
print(f"  • {R2_BUCKET_NAME}/fantasai/injuries/silver_player_news.json")

print("\n🎯 Frontend Ready:")
print("  • api.r2.combinedPlayerNews() - ⭐ Use this! (all data + 3 timestamps)")
print("  • api.r2.playerNotes() - player summaries only")
print("  • api.r2.criticalAlerts() - AI summaries only")
print("  • api.r2.playerInjuries() - injury status only")
print("\n📅 Timestamps in Combined Endpoint:")
print("  • published_at: Original article publication date")
print("  • enriched_at: When article was processed by our system")
print("  • ai_generated_at: When AI summary was generated")

print("\n⏱️  Exported at:", datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"))
print("=" * 70)
print("\n🚀 Databricks → R2 → Frontend pipeline complete!")